# NB_Silver_SCD2
Cleans one day of Bronze into `silver.<resource>`:

1. Flattens the key FHIR fields into typed columns (mapping dictionary below).
2. Deduplicates: keeps the latest version of each `id` within the day (`meta.lastUpdated`, then `extraction_timestamp`).
3. **SCD Type 2**: a hash of the business columns (`hash_diff`) is compared with the current Silver row.
   * new `id` -> inserted as current
   * changed hash -> old row closed (`is_current = false`, `effective_to`), new version inserted
   * same hash -> nothing happens (so re-running a day creates no duplicate versions)

In [ ]:
# Parameters (injected by the pipeline)
resource_name = "Condition"   # Patient / Encounter / Observation / Condition
load_date     = "2026-11-03"  # YYYY-MM-DD
run_id        = ""

In [ ]:

# ======================================================================================
# CELL 1: SILVER RESOURCE CONFIGURATION # Patient / Encounter / Observation / Condition
# ======================================================================================


import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

# Dynamic resource names
resource_lower = resource_name.lower()
resource_upper = resource_name.upper()

# Generate Silver processing run ID
run_id = run_id or str(uuid.uuid4())

# Dynamic Bronze and Silver tables
src = f"bronze.{resource_lower}"
tgt = f"silver.{resource_lower}"

# Current processing timestamp
now_utc = lambda: datetime.now(timezone.utc).replace(tzinfo=None)

print("Resource:", resource_name)
print("Load date:", load_date)
print("Bronze source:", src)
print("Silver target:", tgt)
print("Silver Run ID:", run_id)


In [40]:

# ============================================
# CELL 2: LOGGING FUNCTIONS AND FHIR MAPPINGS
# ============================================

# Logging schema
LOG_SCHEMA = (
    "run_id string, resource_name string, load_date string, "
    "layer string, step string, page_no int, api_url string, "
    "api_call_time timestamp, http_status int, rows_fetched int, "
    "rows_written long, api_total long, raw_file_path string, "
    "data_saved_time timestamp, status string, message string"
)


# ============================================
# LOGGING FUNCTIONS
# ============================================

def log_rows(rows):

    df_log = spark.createDataFrame(rows, LOG_SCHEMA)

    df_log = df_log.select(
        F.col("run_id").cast("string"),
        F.col("resource_name").cast("string"),
        F.col("layer").cast("string"),
        F.col("page_no").cast("int"),
        F.col("api_url").cast("string"),
        F.col("api_call_time").cast("timestamp"),
        F.col("http_status").cast("int"),
        F.col("rows_fetched").cast("int"),
        F.col("raw_file_path").cast("string"),
        F.col("data_saved_time").cast("timestamp"),
        F.col("status").cast("string"),
        F.col("message").cast("string").alias("error_message"),
        F.col("load_date").cast("string"),
        F.col("step").cast("string"),
        F.col("rows_written").cast("long"),
        F.col("api_total").cast("long")
    )

    df_log.write.format("delta").mode("append").saveAsTable(
        "meta.pipeline_log"
    )


def log_summary(
    layer,
    fetched,
    written,
    status="SUCCESS",
    message=None,
    api_total=None,
    api_url=None,
    path=None
):

    log_rows([(
        run_id,
        resource_name,
        load_date,
        layer,
        "summary",
        None,
        api_url,
        None,
        None,
        fetched,
        written,
        api_total,
        path,
        now_utc(),
        status,
        message
    )])


# ============================================
# FHIR RESOURCE COLUMN MAPPINGS
# ============================================

J = lambda path: (
    f"get_json_object(resource_json, '{path}')"
)

REF = lambda path: (
    f"regexp_extract({J(path)}, '([^/]+)$', 1)"
)


MAPPINGS = {

    "Patient": {
        "gender": J("$.gender"),
        "birth_date": J("$.birthDate"),
        "family_name": J("$.name[0].family"),
        "given_name": J("$.name[0].given[0]"),
        "city": J("$.address[0].city"),
        "state": J("$.address[0].state"),
        "country": J("$.address[0].country")
    },

    "Encounter": {
        "status": J("$.status"),
        "class_code": J("$.class.code"),
        "patient_id": REF("$.subject.reference"),
        "period_start": J("$.period.start"),
        "period_end": J("$.period.end")
    },

    "Observation": {
        "status": J("$.status"),
        "code": J("$.code.coding[0].code"),
        "code_display": J("$.code.coding[0].display"),
        "value_number": (
            f"cast({J('$.valueQuantity.value')} as double)"
        ),
        "value_unit": J("$.valueQuantity.unit"),
        "patient_id": REF("$.subject.reference"),
        "encounter_id": REF("$.encounter.reference"),
        "effective_datetime": J("$.effectiveDateTime")
    },

    "Condition": {
        "clinical_status": J("$.clinicalStatus.coding[0].code"),
        "code": J("$.code.coding[0].code"),
        "code_display": J("$.code.coding[0].display"),
        "patient_id": REF("$.subject.reference"),
        "encounter_id": REF("$.encounter.reference"),
        "onset_datetime": J("$.onsetDateTime"),
        "recorded_date": J("$.recordedDate")
    }
}


# ============================================
# VALIDATE RESOURCE MAPPING
# ============================================

if resource_name not in MAPPINGS:
    raise ValueError(
        f"Unsupported resource: {resource_name}"
    )

mapping = MAPPINGS[resource_name]

load_ts = now_utc()

print(f"{resource_upper} mapping loaded successfully.")
print("Mapped columns:", list(mapping.keys()))


StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 42, Finished, Available, Finished, False)

CONDITION mapping loaded successfully.
Mapped columns: ['clinical_status', 'code', 'code_display', 'patient_id', 'encounter_id', 'onset_datetime', 'recorded_date']


In [41]:

# ============================================
# CELL 3: READ BRONZE DATA
# ============================================

# Find the latest successful Bronze run
bronze_log = (
    spark.table("meta.pipeline_log")
    .filter(
        (F.col("resource_name") == resource_name) &
        (F.col("load_date") == load_date) &
        (F.col("layer") == "bronze") &
        (F.col("step") == "summary") &
        (F.col("status") == "SUCCESS")
    )
    .orderBy(F.col("data_saved_time").desc())
)

bronze_record = bronze_log.first()

if bronze_record is None:
    raise RuntimeError(
        f"No successful Bronze run found for {resource_name} "
        f"on {load_date}."
    )

bronze_run_id = bronze_record["run_id"]
expected_bronze_rows = bronze_record["rows_written"]

if not bronze_run_id or expected_bronze_rows is None:
    raise RuntimeError("Bronze run ID or row count is missing.")

# Read selected Bronze run
b = (
    spark.table(src)
    .filter(
        (F.col("run_id") == bronze_run_id) &
        (F.col("ingest_date") == load_date)
    )
    .withColumn(
        "resource_json",
        F.to_json(F.col("resource"))
    )
)

n_bronze = b.count()

print("Resource:", resource_name)
print("Bronze run ID:", bronze_run_id)
print("Expected Bronze records:", expected_bronze_rows)
print("Actual Bronze records:", n_bronze)

if n_bronze != expected_bronze_rows:
    raise RuntimeError("Bronze record count mismatch.")

print(f"{resource_name} Bronze data read successfully.")


StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 43, Finished, Available, Finished, False)

Resource: Condition
Bronze run ID: 660f0af9-ac1a-48f0-812b-989c7c3ba0c1
Expected Bronze records: 989
Actual Bronze records: 989
Condition Bronze data read successfully.


In [42]:

# ============================================
# CELL 4: FLATTEN FHIR RESOURCES
# ============================================

flat = (
    b.select(
        F.col("resource_id").alias("id"),
        *[
            F.expr(expression).alias(column)
            for column, expression in mapping.items()
        ],
        F.col("source_last_updated")
            .cast("timestamp").alias("source_last_updated"),
        F.col("extraction_timestamp"),
        F.col("api_url_or_params"),
        F.col("ingest_date"),
        F.col("run_id")
    )
    .filter(F.col("id").isNotNull())
)

print(f"{resource_name} flattened records:", flat.count())

print("Silver columns:")
flat.printSchema()

display(flat.limit(10))


StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 44, Finished, Available, Finished, False)

Condition flattened records: 989
Silver columns:
root
 |-- id: string (nullable = true)
 |-- clinical_status: string (nullable = true)
 |-- code: string (nullable = true)
 |-- code_display: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- onset_datetime: string (nullable = true)
 |-- recorded_date: string (nullable = true)
 |-- source_last_updated: timestamp (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- api_url_or_params: string (nullable = true)
 |-- ingest_date: string (nullable = true)
 |-- run_id: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 77b28b46-f1a4-4f1a-9799-28e6646f7a3e)

In [43]:

# ============================================
# CELL 5: DEDUPLICATE FHIR RESOURCES
# ============================================

window_spec = (
    Window.partitionBy("id")
    .orderBy(
        F.col("source_last_updated").desc_nulls_last(),
        F.col("extraction_timestamp").desc_nulls_last()
    )
)

dedup = (
    flat
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

n_flat = flat.count()
n_dedup = dedup.count()

duplicate_count = n_flat - n_dedup

print(f"{resource_name} records before deduplication:", n_flat)
print(f"{resource_name} records after deduplication:", n_dedup)
print("Duplicate records removed:", duplicate_count)

if dedup.filter(F.col("id").isNull()).count() > 0:
    raise RuntimeError("Null resource IDs found.")

if dedup.groupBy("id").count().filter(
    F.col("count") > 1
).count() > 0:
    raise RuntimeError("Duplicate IDs remain after deduplication.")

print(f"{resource_name} deduplication successful.")

display(dedup.limit(10))


StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 45, Finished, Available, Finished, False)

Condition records before deduplication: 989
Condition records after deduplication: 989
Duplicate records removed: 0
Condition deduplication successful.


SynapseWidget(Synapse.DataFrame, 257eb23f-2fa9-4fc4-9ed6-f1f7ae513d10)

In [44]:

# ============================================
# CELL 6: GENERATE HASH AND SCD2 COLUMNS
# ============================================

business_columns = list(mapping.keys())

new_rows = (
    dedup
    .withColumn(
        "hash_diff",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column).cast("string"),
                        F.lit("~")
                    )
                    for column in business_columns
                ]
            ),
            256
        )
    )
    .withColumn(
        "effective_from",
        F.lit(load_ts).cast("timestamp")
    )
    .withColumn(
        "effective_to",
        F.lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        F.lit(True)
    )
).cache()

n_new_rows = new_rows.count()

print(f"{resource_name} SCD2 input records:", n_new_rows)
print("Business columns:", business_columns)
print("SCD2 columns generated successfully.")

display(new_rows.limit(10))


StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 46, Finished, Available, Finished, False)

Condition SCD2 input records: 989
Business columns: ['clinical_status', 'code', 'code_display', 'patient_id', 'encounter_id', 'onset_datetime', 'recorded_date']
SCD2 columns generated successfully.


SynapseWidget(Synapse.DataFrame, a69c30fd-162c-4b99-b78e-2a1e39c215e3)

In [45]:
# ============================================
# CELL 7: CHECK EXISTING SILVER TABLE
# ============================================

target_exists = spark.catalog.tableExists(tgt)

print("Resource:", resource_name)
print("Silver target:", tgt)
print("Target table exists:", target_exists)

if target_exists:
    target_df = spark.table(tgt)

    print("Existing Silver record count:", target_df.count())

    print("Existing Silver schema:")
    target_df.printSchema()

    target_columns = set(target_df.columns)
    incoming_columns = set(new_rows.columns)

    missing_columns = incoming_columns - target_columns
    extra_columns = target_columns - incoming_columns

    print("Columns missing in Silver:", sorted(missing_columns))
    print("Additional columns in Silver:", sorted(extra_columns))

else:
    print("Silver table does not exist. It will be created during the load.")

StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 47, Finished, Available, Finished, False)

Resource: Condition
Silver target: silver.condition
Target table exists: True
Existing Silver record count: 1824
Existing Silver schema:
root
 |-- id: string (nullable = true)
 |-- clinical_status: string (nullable = true)
 |-- code: string (nullable = true)
 |-- code_display: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- onset_datetime: string (nullable = true)
 |-- recorded_date: string (nullable = true)
 |-- source_last_updated: timestamp (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- api_url_or_params: string (nullable = true)
 |-- ingest_date: string (nullable = true)
 |-- hash_diff: string (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)

Columns missing in Silver: ['run_id']
Additional columns in Silver: []


In [46]:
# ============================================
# CELL 8: ADD MISSING SILVER COLUMNS
# ============================================

target_columns = spark.table(tgt).columns

if "run_id" not in target_columns:
    spark.sql(f"""
        ALTER TABLE {tgt}
        ADD COLUMNS (run_id STRING)
    """)
    print("run_id column added successfully.")
else:
    print("run_id column already exists.")

print("Updated Silver schema:")
spark.table(tgt).printSchema()

StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 48, Finished, Available, Finished, False)

run_id column added successfully.
Updated Silver schema:
root
 |-- id: string (nullable = true)
 |-- clinical_status: string (nullable = true)
 |-- code: string (nullable = true)
 |-- code_display: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- onset_datetime: string (nullable = true)
 |-- recorded_date: string (nullable = true)
 |-- source_last_updated: timestamp (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- api_url_or_params: string (nullable = true)
 |-- ingest_date: string (nullable = true)
 |-- hash_diff: string (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- run_id: string (nullable = true)



In [47]:
# ============================================
# CELL 9: IDENTIFY SCD2 CHANGES
# ============================================

target_df = spark.table(tgt)

current_records = (
    target_df
    .filter(F.col("is_current") == True)
    .select("id", "hash_diff")
)

comparison = (
    new_rows.alias("s")
    .join(
        current_records.alias("t"),
        on="id",
        how="left"
    )
)

new_ids = comparison.filter(
    F.col("t.hash_diff").isNull()
)

changed_ids = comparison.filter(
    F.col("t.hash_diff").isNotNull() &
    (F.col("s.hash_diff") != F.col("t.hash_diff"))
)

unchanged_ids = comparison.filter(
    F.col("s.hash_diff") == F.col("t.hash_diff")
)

new_count = new_ids.count()
changed_count = changed_ids.count()
unchanged_count = unchanged_ids.count()

print("Resource:", resource_name)
print("New records:", new_count)
print("Changed records:", changed_count)
print("Unchanged records:", unchanged_count)

print("SCD2 change identification completed.")

StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 49, Finished, Available, Finished, False)

Resource: Condition
New records: 0
Changed records: 0
Unchanged records: 989
SCD2 change identification completed.


In [48]:
# ============================================
# CELL 10: APPLY SCD2 MERGE
# ============================================

if new_count == 0 and changed_count == 0:
    print(f"No new or changed {resource_name} records.")
    print("Silver table remains unchanged.")

else:
    delta_target = DeltaTable.forName(spark, tgt)

    # Close previous versions of changed records
    changed_source = (
        new_rows.alias("s")
        .join(
            current_records.alias("t"),
            on="id",
            how="inner"
        )
        .filter(F.col("s.hash_diff") != F.col("t.hash_diff"))
        .select("s.*")
    )

    if changed_count > 0:
        (
            delta_target.alias("t")
            .merge(
                changed_source.alias("s"),
                "t.id = s.id AND t.is_current = true"
            )
            .whenMatchedUpdate(set={
                "effective_to": "s.effective_from",
                "is_current": "false"
            })
            .execute()
        )

    # Insert new and changed versions
    rows_to_insert = (
        new_rows.alias("s")
        .join(
            current_records.alias("t"),
            on="id",
            how="left"
        )
        .filter(
            F.col("t.hash_diff").isNull() |
            (F.col("s.hash_diff") != F.col("t.hash_diff"))
        )
        .select("s.*")
    )

    if new_count + changed_count > 0:
        (
            DeltaTable.forName(spark, tgt)
            .alias("t")
            .merge(
                rows_to_insert.alias("s"),
                "t.id = s.id AND t.is_current = true"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    print(f"{resource_name} SCD2 merge completed.")

StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 50, Finished, Available, Finished, False)

No new or changed Condition records.
Silver table remains unchanged.


In [49]:
# ============================================
# CELL 11: VALIDATE SILVER DATA
# ============================================

silver_df = spark.table(tgt)

total_records = silver_df.count()

current_records_count = (
    silver_df
    .filter(F.col("is_current") == True)
    .count()
)

historical_records_count = (
    silver_df
    .filter(F.col("is_current") == False)
    .count()
)

duplicate_current_ids = (
    silver_df
    .filter(F.col("is_current") == True)
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Resource:", resource_name)
print("Silver table:", tgt)
print("Total Silver records:", total_records)
print("Current records:", current_records_count)
print("Historical records:", historical_records_count)
print("Duplicate current IDs:", duplicate_current_ids)

if duplicate_current_ids > 0:
    raise RuntimeError("Duplicate current records found.")

print(f"{resource_name} Silver validation successful.")

display(
    silver_df
    .filter(F.col("is_current") == True)
    .limit(10)
)

StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 51, Finished, Available, Finished, False)

Resource: Condition
Silver table: silver.condition
Total Silver records: 1824
Current records: 1611
Historical records: 213
Duplicate current IDs: 0
Condition Silver validation successful.


SynapseWidget(Synapse.DataFrame, 7bf20fe9-977b-42a1-ab6c-04722f5beca4)

In [50]:
# ============================================
# CELL 12: LOG SILVER PROCESSING RESULT
# ============================================

log_summary(
    layer="silver",
    fetched=n_bronze,
    written=new_count + changed_count,
    status="SUCCESS",
    message=(
        f"{resource_name} SCD2 processing completed. "
        f"New: {new_count}, Changed: {changed_count}, "
        f"Unchanged: {unchanged_count}, "
        f"Total Silver records: {total_records}"
    ),
    api_total=expected_bronze_rows
)

print(f"{resource_name} Silver processing logged successfully.")
print("Silver processing run ID:", run_id)

StatementMeta(, ebe098fe-a61a-487e-863e-e5f4bd8dd6d1, 52, Finished, Available, Finished, False)

Condition Silver processing logged successfully.
Silver processing run ID: b77b6acc-3c0f-43e5-b642-4326046e4046
